# Lid-driven cavity

Steady incompressible Navier-Stokes at $\mathrm{Re}=100$ on $[0,1]^2$:

$$\mathbf{u}\!\cdot\!\nabla\mathbf{u} + \nabla p - \tfrac{1}{\mathrm{Re}}\nabla^2\mathbf{u}=0,\qquad \nabla\!\cdot\!\mathbf{u}=0.$$

The lid ($y=1$) moves with $u=4x(1-x)$; the other walls are no-slip.
Needs a reference field at `pinn/references/ldc.npz` (generate with `ldc/reference_gen/make_ldc_reference.py`).


In [ ]:
import os, pathlib, sys, tempfile, traceback, importlib.util


def _repo_root(start=None):
    """Walk up from the kernel's cwd (or the hub layouts) to the checkout root."""
    starts = [pathlib.Path(start or pathlib.Path.cwd()).resolve(),
              pathlib.Path.home() / "serafin" / "Julia" / "PINN",   # hub node (JupyterLab)
              pathlib.Path.home() / "Julia" / "PINN"]               # login node
    for base in starts:
        for p in (base, *base.parents):
            if (p / "Stationary" / "stationary" / "dsgnar.py").exists():
                return p
    raise RuntimeError("repo root not found (no Stationary/stationary/dsgnar.py above cwd or $HOME)")


REPO = _repo_root()
# the paper's OWN package (github.com/wephy/physics-informed-neural-networks), vendored here
sys.path.insert(0, str(REPO / "Evolution_try" / "vendor" / "dsgnar_pinn"))

import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")   # ~/.matplotlib is not writable here
# must be set before the first `import jax` in this kernel, else it grabs 75% of the hub GPU
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
# The hub GPU is shared through HAMi, and XLA's autotuner allocates large temporary
# workspaces per candidate (measured: an 8 GiB request on a 19.3 GiB slice).  Turn
# autotuning off; for this small network it buys little and costs GBs.
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_autotune_level=0")
# XLA's compilation/autotune cache writes a temp file and renames it; on the shared network
# tree that fails with EIO ("Failed to insert autotune cache: ... Input/output error").
# Keep it node-local, as COMMANDS.md section 3 does for every hub run.
CACHE_DIR = pathlib.Path(tempfile.gettempdir()) / f"jaxcache-{os.environ.get('USER', 'user')}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── startup diagnostic -------------------------------------------------------------
# Writes the environment and any import traceback to ldc/diag.txt on the shared tree, so a
# failure can be read from anywhere (the Mac included) even if the notebook is not saved.
_REQUIRED = ("jax", "equinox", "jaxtyping", "matplotlib", "IPython", "ipywidgets")
_found = {m: importlib.util.find_spec(m) is not None for m in _REQUIRED}
_missing = [m for m, ok in _found.items() if not ok]
_lines = [f"python {sys.version.split()[0]}",
          f"cwd    {pathlib.Path.cwd()}",
          f"repo   {REPO}",
          f"cache  {CACHE_DIR}",
          f"found  {_found}"]
_error = None
try:
    import jax
    import jax.numpy as jnp
    _lines.append(f"jax    {jax.__version__}  backend={jax.default_backend()}  "
                  f"devices={[str(d) for d in jax.devices()]}")
except Exception:
    _error = traceback.format_exc()
    _lines.append("jax IMPORT FAILED\n" + _error)
try:
    import pinn
    from pinn import operators as op, sampling
    _lines.append(f"pinn   {getattr(pinn, '__file__', '?')}")
except Exception:
    _error = traceback.format_exc()
    _lines.append("pinn IMPORT FAILED\n" + _error)

_DIAG = REPO / "ldc" / "diag.txt"
try:
    _DIAG.write_text("\n".join(_lines) + "\n")
except Exception:
    pass

if _missing or _error is not None:
    raise RuntimeError(
        "the paper notebook cannot start.\n"
        f"missing packages: {_missing or 'none'}\n"
        f"full report written to {_DIAG}\n\n"
        "If packages are missing, install them in THIS kernel's environment (JupyterLab terminal):\n"
        "    cd ~/serafin/Julia/PINN/Stationary\n"
        "    .venv/bin/python -m pip install equinox jaxtyping ipython ipywidgets\n"
        "then Kernel -> Restart Kernel and run this cell again.\n\n"
        "Report so far:\n" + "\n".join(_lines[-8:]))

%matplotlib inline
jax.config.update("jax_enable_x64", True)   # double precision
print(f"pid   {os.getpid()}   <- a fresh kernel gets a NEW pid; check it against the log")
print(f"flags XLA_FLAGS={os.environ.get('XLA_FLAGS')}")
print(f"repo  {REPO}")
print(f"cache {CACHE_DIR}   (node-local; the shared tree rejects XLA's cache writes)")
print(f"jax {jax.__version__} | backend {jax.default_backend()} | "
      f"{[str(d) for d in jax.devices()]}")
if jax.default_backend() != "gpu":
    print("warning: this is a CPU kernel -- run with SMOKE=True, or pick 'PINN (jax, CUDA)'")


In [ ]:
class LidDrivenCavityProblem(pinn.Problem):
    """Steady lid-driven cavity at Re = 100; outputs (u, v, p)."""

    x_min, x_max = 0.0, 1.0
    Re = 100.0
    problem_name = "LidDrivenCavity"
    ref_path = str(REPO / "ldc" / "reference_gen" / "ldc_reference.npz")

    def __init__(self, *, n_pde, n_bc, n_ic=0):
        self.n_pde, self.n_bc, self.n_ic = n_pde, n_bc, n_ic

    def residual_fns(self):
        return {"pde": self.pde_residual, "bc": self.bc_residual}

    def pde_residual(self, model, coords):
        x, y = coords[0], coords[1]
        u_ = lambda c: model(c)[0]
        v_ = lambda c: model(c)[1]
        p_ = lambda c: model(c)[2]
        u, v = u_(coords), v_(coords)
        inv_Re = 1.0 / self.Re
        res_u = (u * op.grad(u_, coords, 0) + v * op.grad(u_, coords, 1)
                 + op.grad(p_, coords, 0)
                 - inv_Re * (op.grad2(u_, coords, 0) + op.grad2(u_, coords, 1)))
        res_v = (u * op.grad(v_, coords, 0) + v * op.grad(v_, coords, 1)
                 + op.grad(p_, coords, 1)
                 - inv_Re * (op.grad2(v_, coords, 0) + op.grad2(v_, coords, 1)))
        res_div = op.grad(u_, coords, 0) + op.grad(v_, coords, 1)
        return jnp.array([res_u, res_v, 2.0 * res_div])

    def bc_residual(self, model, coords):
        x, y = coords[0], coords[1]
        u, v = model(coords)[0], model(coords)[1]
        u_target = jnp.where(jnp.isclose(y, 1.0), 4.0 * x * (1.0 - x), 0.0)
        return jnp.array([u - u_target, v, 0.0])

    def samplers(self):
        return {"pde": self._sample_pde, "bc": self._sample_bc}

    def _sample_pde(self, key, n, *_):
        # 7/8 bulk points, 1/16 near the moving lid, 1/16 in the top corners.
        n_lid, n_corner = n // 16, n // 16
        n_bulk = n - n_lid - n_corner
        k_bulk, k_lx, k_ly, k_l, k_r = jax.random.split(key, 5)
        bulk = jax.random.uniform(k_bulk, (n_bulk, 2), minval=0.0, maxval=1.0)
        lid = jnp.stack([
            jax.random.uniform(k_lx, (n_lid,)),
            jax.random.uniform(k_ly, (n_lid,), minval=0.9, maxval=1.0)], axis=1)
        half = n_corner // 2
        s1 = jax.random.uniform(k_l, (half, 2))
        left = jnp.stack([0.1 * s1[:, 0]**2, 1.0 - 0.1 * s1[:, 1]**2], axis=1)
        s2 = jax.random.uniform(k_r, (half, 2))
        right = jnp.stack([1.0 - 0.1 * s2[:, 0]**2, 1.0 - 0.1 * s2[:, 1]**2], axis=1)
        return jnp.concatenate([bulk, lid, left, right], axis=0)

    def _sample_bc(self, key, n, *_):
        n_corner = n // 16
        n_uniform = n - n_corner
        half = n_corner // 2
        k_pt, k_dim, k_face, k_l, k_r = jax.random.split(key, 5)
        pts = jax.random.uniform(k_pt, (n_uniform, 2), minval=0.0, maxval=1.0)
        face_dim = jax.random.randint(k_dim, (n_uniform,), 0, 2)
        face_val = jax.random.randint(k_face, (n_uniform,), 0, 2).astype(pts.dtype)
        pts = pts.at[jnp.arange(n_uniform), face_dim].set(face_val)
        left = jnp.stack([0.1 * jax.random.uniform(k_l, (half,)),
                          jnp.ones((half,))], axis=1)
        right = jnp.stack([1.0 - 0.1 * jax.random.uniform(k_r, (half,)),
                           jnp.ones((half,))], axis=1)
        return jnp.concatenate([pts, left, right], axis=0)

In [ ]:
# SMOKE = True is a few-minute CPU sanity check; set False for the production run
# (2**15 PDE points, 4000-wide sketches, 60-60-60-60 SIREN -> hours on CPU).
SMOKE = True

cfg = pinn.RunConfig(
    network=lambda key: pinn.SIREN(
        key, LidDrivenCavityProblem, time_dependent=False,
        hidden_dims=(40, 40, 40) if SMOKE else (60, 60, 60, 60),
        periodic_bc=False, n_inputs=2, n_outputs=3,
    ),
    # the paper's package builds a DENSE Jacobian per macro-batch (jacrev), so batch_size
    # and the point count set its peak memory; on the shared 19.3 GiB A30 keep the smoke
    # run small, and start the production config only on a bigger allocation
    n_pde=2**11 if SMOKE else 2**15,
    n_bc=2**10 if SMOKE else 2**14,
    residual_sketch=256 if SMOKE else 4000,
    parameter_sketch=256 if SMOKE else 4000,
    batch_size=2**8 if SMOKE else 2**12,
    sub_batch_size=4,
    probe_batch_size=2**7,
    cache_dir=str(CACHE_DIR),   # node-local: the shared tree rejects XLA's cache writes
    pde_weight=1e-2,
    min_radius=1e-2 if SMOKE else None,   # bounds the float64 pass so it terminates
    float32_steps=2 if SMOKE else 10,
)


In [ ]:
# Compile-cache warmer: it compiles the same graphs train64 uses, so it is worth
# running once for a production run, but it costs minutes -> skipped in SMOKE mode.
if not SMOKE:
    pinn.precompile64(LidDrivenCavityProblem, cfg)


In [ ]:
results = pinn.train64(LidDrivenCavityProblem, cfg)

## Results

The pressure field is defined only up to a constant, so the error panel compares it after removing the mean.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ref = pinn.Reference.load(str(REPO / "ldc" / "reference_gen" / "ldc_reference.npz"))   # the paper package has no ldc.npz of its own
x0, x1 = results["plot_x0"], results["plot_x1"]
lx, ly = results["plot_axes"]
extent = [x0[0], x0[-1], x1[0], x1[-1]]

for ch in ref.channels:
    pred = np.array(results["u_pred_plot"][ch])
    exact = np.array(ref.plot_grids[ch])
    err = np.abs(pred - exact)
    rel = np.linalg.norm(pred - exact) / np.linalg.norm(exact)

    fig, axs = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, data, title, cmap in zip(
        axs, [pred, exact, err],
        [f"PINN  ${ch}$", f"reference  ${ch}$", f"abs error  (rel $\\ell_2$={rel:.2e})"],
        ["RdBu_r", "RdBu_r", "magma"],
    ):
        im = ax.imshow(data.T, origin="lower", aspect="auto", extent=extent, cmap=cmap)
        ax.set(xlabel=f"${lx}$", ylabel=f"${ly}$", title=title)
        fig.colorbar(im, ax=ax)
    plt.show()